# 🧊 Cold Chain Data Exploration

This notebook connects to InfluxDB to analyze historical sensor data, calculate Z-scores for anomaly detection, and visualize logistics trends.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from influxdb_client import InfluxDBClient
from dotenv import load_dotenv

# 1. LOAD CONFIGURATION
load_dotenv("../.env")

INFLUX_URL = os.getenv("INFLUXDB_URL", "http://localhost:8086")
INFLUX_TOKEN = os.getenv("INFLUXDB_TOKEN")
INFLUX_ORG = os.getenv("INFLUXDB_ORG")
INFLUX_BUCKET = os.getenv("INFLUXDB_BUCKET")

print("Configuration Loaded.")

## 2. Connect to InfluxDB

In [ ]:
client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
query_api = client.query_api()
print("Connection established.")

## 3. Fetch Historical Data (Last 24 Hours)

In [ ]:
query = f'''
    from(bucket: "{INFLUX_BUCKET}")
    |> range(start: -24h)
    |> filter(fn: (r) => r._measurement == "sensor_reading")
    |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
'''

result = query_api.query_data_frame(query)

# Handle potential list of dataframes from InfluxDB
if isinstance(result, list):
    df = pd.concat(result, ignore_index=True)
else:
    df = result

if not df.empty:
    df["_time"] = pd.to_datetime(df["_time"])
    print(f"Success! Loaded {len(df)} records.")
    display(df.head())
else:
    print("No data found.")

## 4. Basic Exploration

In [ ]:
print("--- Summary Statistics ---")
display(df.describe())

print("--- Average Temperature per Sensor ---")
if "sensor_id" in df.columns:
    display(df.groupby("sensor_id")["temperature_c"].mean())

## 5. Visualizing Trends (1-Minute Resampling)

In [ ]:
plt.figure(figsize=(12, 6))
df.set_index("_time").resample("1min")["temperature_c"].mean().plot()
plt.title("1-Minute Resampled Temperature Trend")
plt.ylabel("Temp (C)")
plt.grid(True)
plt.show()

## 6. Z-Score Anomaly Detection

Here we flag any reading that is more than 2 standard deviations away from the mean.

In [ ]:
df["z_score"] = (df["temperature_c"] - df["temperature_c"].mean()) / df["temperature_c"].std()

# Flag readings more than 2 std deviations from mean
anomalies = df[df["z_score"].abs() > 2]

print(f"Detected {len(anomalies)} anomalies.")
if not anomalies.empty:
    display(anomalies[["_time", "sensor_id", "temperature_c", "z_score"]].head())
    
    # Plotting anomalies
    plt.figure(figsize=(12, 6))
    plt.plot(df["_time"], df["temperature_c"], label="Actual")
    plt.scatter(anomalies["_time"], anomalies["temperature_c"], color='red', label="Anomaly")
    plt.legend()
    plt.title("Anomaly Detection (Z-Score > 2)")
    plt.show()